# Day 4 — LLM-Based Tool Calling: Web Search + Summarization

**What we'll build:** A tool-calling system where the LLM decides which tool to invoke — web search or summarization — based on the user's query.

**What you'll learn:**
- How to define tools using LangChain's `@tool` decorator
- How tool calling works — the LLM picks the tool, your code executes it
- Two approaches: a manual JSON loop and a ReAct agent

## 1. Setup & Installation

In [ ]:
!pip install -q langchain langchain-ollama langchain-community tavily-python

In [ ]:
import os
import json
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from dotenv import load_dotenv

load_dotenv()

# Set your Tavily API key (get one free at https://tavily.com)
# os.environ["TAVILY_API_KEY"] = "your-tavily-api-key-here"

#TAVILY_API_KEY = your-tavily-api-key-here

c:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## 2. Initialize the LLM

We use `llama3.2:3b` via Ollama. Make sure Ollama is running and the model is pulled.

In [3]:
llm = ChatOllama(model="llama3.2:3b", temperature=0)

# Quick test — make sure the LLM responds
response = llm.invoke("Say hello in one sentence.")
print(response.content)

Hello!


## 3. Define Tool 1 — Web Search

We use the `@tool` decorator to define our first tool. The **docstring becomes the tool description** — this is what the LLM reads to decide when to use it.

In [4]:
@tool
def web_search(query: str) -> str:
    """Search the web for real-time information on any topic."""
    search = TavilySearchResults(max_results=3)
    results = search.invoke({"query": query})
    # Format results into a readable string
    output = ""
    for r in results:
        output += f"- {r['content']}\n\n"
    return output.strip()

**Test it standalone** — see what the tool returns before any agent uses it.

In [5]:
result = web_search.invoke("latest news on OpenAI")
print(result[:500])

C:\Users\shiva\AppData\Local\Temp\ipykernel_14364\3423942845.py:4: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search = TavilySearchResults(max_results=3)


- Our Research. Research Index · Research Overview ; Latest Advancements. GPT-5.3 Instant · GPT-5.3-Codex ; Safety. Safety Approach · Security & Privacy

- Introducing GPT-5.3-Codex-Spark—our first real-time coding model. 15x faster generation, 128k context, now in research preview for ChatGPT Pro users. Product.

- We're rolling out updated Box, Notion, Linear, and Dropbox apps in ChatGPT. These updates add new app actions, including new write capabilities where supported,


## 4. Define Tool 2 — Summarize

This tool takes a long text and returns a short summary using the LLM.

In [6]:
@tool
def summarize(text: str) -> str:
    """Summarize a long piece of text into a short paragraph."""
    response = llm.invoke(f"Summarize the following text in 2-3 sentences:\n\n{text}")
    return response.content

**Test it standalone** — pass a paragraph and check the summary.

In [7]:
sample_text = """
Artificial intelligence has seen rapid advancements in recent years,
particularly in the field of large language models. Companies like OpenAI,
Google, and Meta have released increasingly capable models that can understand
and generate human-like text. These models are being used in applications
ranging from customer service chatbots to code generation tools, transforming
industries and raising important questions about the future of work and
the ethical implications of AI technology.
"""

result = summarize.invoke(sample_text)
print(result)

Here is a summary of the text in 2-3 sentences:

Artificial intelligence has made significant progress in recent years, particularly in large language models, which can understand and generate human-like text. Companies like OpenAI, Google, and Meta have released increasingly capable models that are being used in various applications, including customer service chatbots and code generation tools. This has raised important questions about the future of work and the ethical implications of AI technology.


## 5. Approach 1 — Manual Agent Loop

This is the core lesson. We build the tool-calling loop from scratch:

1. Tell the LLM what tools are available (via system prompt)
2. LLM responds with a JSON tool call
3. Our code parses the JSON and executes the tool
4. Feed the result back to the LLM for a final answer

This is exactly what happens inside frameworks like LangChain — we're just doing it manually so you can see every step.

In [8]:
SYSTEM_PROMPT = """You are a helpful assistant with access to two tools:

1. web_search(query: str) — Search the web for real-time information on any topic.
2. summarize(text: str) — Summarize a long piece of text into a short paragraph.

When the user asks a question, decide which tool to use.

RULES:
- If you need to call a tool, respond ONLY with valid JSON in this exact format:
  {{"tool": "tool_name", "args": {{"param": "value"}}}}
- If you already have enough information to answer, respond ONLY with:
  {{"tool": "final_answer", "args": {{"answer": "your answer here"}}}}
- Do NOT include any text outside the JSON.
- For queries that need search AND summarization, call web_search first.
"""

Now the loop — this function runs the full agent cycle.

In [9]:
# Map tool names to actual functions
tools_map = {
    "web_search": web_search,
    "summarize": summarize,
}

def run_agent(user_query, max_steps=3):
    """Run the manual tool-calling loop."""
    print(f"\n{'='*60}")
    print(f"User Query: {user_query}")
    print(f"{'='*60}")

    messages = [
        ("system", SYSTEM_PROMPT),
        ("human", user_query),
    ]

    for step in range(max_steps):
        # Ask the LLM what to do
        response = llm.invoke(messages)
        raw = response.content.strip()

        # Parse the JSON response
        try:
            # Clean up if LLM wraps in markdown code blocks
            if raw.startswith("```"):
                raw = raw.split("```")[1]
                if raw.startswith("json"):
                    raw = raw[4:]
                raw = raw.strip()
            decision = json.loads(raw)
        except json.JSONDecodeError:
            print(f"\n[Step {step+1}] LLM returned non-JSON, treating as final answer.")
            print(f"Answer: {raw}")
            return raw

        tool_name = decision.get("tool", "")
        args = decision.get("args", {})

        # If LLM says it has the final answer, we're done
        if tool_name == "final_answer":
            answer = args.get("answer", raw)
            print(f"\n[Step {step+1}] Final Answer")
            print(f"Answer: {answer}")
            return answer

        # Otherwise, execute the tool
        if tool_name in tools_map:
            print(f"\n[Step {step+1}] Calling tool: {tool_name}")
            print(f"  Args: {args}")

            # Call the tool
            tool_fn = tools_map[tool_name]
            tool_result = tool_fn.invoke(list(args.values())[0])
            print(f"  Result (preview): {tool_result[:200]}...")

            # Feed the result back to the LLM
            messages.append(("assistant", raw))
            messages.append(("human",
                f"Tool '{tool_name}' returned:\n{tool_result}\n\n"
                f"Now either call another tool or provide the final answer as JSON."
            ))
        else:
            print(f"\n[Step {step+1}] Unknown tool: {tool_name}")
            return None

    print("\n[Max steps reached]")
    return None

## 6. Test the Manual Agent Loop

Three test queries — each one exercises a different routing path.

**Test 1:** Should route to `web_search`

In [10]:
run_agent("What is the latest news on OpenAI?")


User Query: What is the latest news on OpenAI?

[Step 1] LLM returned non-JSON, treating as final answer.
Answer: {{"tool": "web_search", "args": {"query": "OpenAI latest news"}}}


'{{"tool": "web_search", "args": {"query": "OpenAI latest news"}}}'

**Test 2:** Should route to `summarize`

In [11]:
long_text = """
Large language models (LLMs) are a type of artificial intelligence that have been
trained on vast amounts of text data. They can generate human-like text, translate
languages, write different kinds of creative content, and answer your questions in
an informative way. Recent developments include models with longer context windows,
improved reasoning capabilities, and the ability to use external tools. Companies
are increasingly integrating LLMs into their products and services, from search
engines to productivity tools, fundamentally changing how people interact with
technology and access information.
"""

run_agent(f"Summarize this paragraph: {long_text}")


User Query: Summarize this paragraph: 
Large language models (LLMs) are a type of artificial intelligence that have been
trained on vast amounts of text data. They can generate human-like text, translate
languages, write different kinds of creative content, and answer your questions in
an informative way. Recent developments include models with longer context windows,
improved reasoning capabilities, and the ability to use external tools. Companies
are increasingly integrating LLMs into their products and services, from search
engines to productivity tools, fundamentally changing how people interact with
technology and access information.


[Step 1] LLM returned non-JSON, treating as final answer.
Answer: {{"tool": "final_answer", "args": {{"answer": "Large language models are AI trained on vast text data, generating human-like text, translating languages, and answering questions, and are being integrated into various products and services."}}}


'{{"tool": "final_answer", "args": {{"answer": "Large language models are AI trained on vast text data, generating human-like text, translating languages, and answering questions, and are being integrated into various products and services."}}}'

**Test 3 (Bonus):** Should call `web_search` first, then `summarize` the results.

In [12]:
run_agent("Find the latest news on AI agents and summarize it")


User Query: Find the latest news on AI agents and summarize it

[Step 1] LLM returned non-JSON, treating as final answer.
Answer: {{"tool": "web_search", "args": {"query": "latest news AI agents"}}}

{{"tool": "summarize", "args": {"text": "AI agents are becoming increasingly sophisticated, with recent advancements in areas such as reinforcement learning and deep learning. Researchers are exploring the use of AI agents in complex tasks such as autonomous vehicles and healthcare. However, there are also concerns about the potential risks and challenges associated with the development of AI agents, including job displacement and bias."}}


'{{"tool": "web_search", "args": {"query": "latest news AI agents"}}}\n\n{{"tool": "summarize", "args": {"text": "AI agents are becoming increasingly sophisticated, with recent advancements in areas such as reinforcement learning and deep learning. Researchers are exploring the use of AI agents in complex tasks such as autonomous vehicles and healthcare. However, there are also concerns about the potential risks and challenges associated with the development of AI agents, including job displacement and bias."}}'

## 7. Approach 2 — ReAct Agent with LangChain

The manual loop above is exactly what LangChain's `create_react_agent` does under the hood. Let's see the same thing in ~5 lines.

**ReAct = Reason + Act** — the LLM thinks step by step, picks a tool, observes the result, and repeats until done.

In [14]:
from langchain.agents import create_agent

# Create the agent with our two tools
tools = [web_search, summarize]
agent = create_agent(llm, tools)

In [17]:
def run_react_agent(query):
    """Run a query through the ReAct agent and print the result."""
    print(f"\nQuery: {query}")
    print("-" * 40)

    result = agent.invoke({"messages": [("human", query)]})

    # Print each step the agent took
    for msg in result["messages"]:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"[Tool Call] {tc['name']}({tc['args']})")
        elif msg.type == "tool":
            print(f"[Tool Result] {msg.content}...")
        elif msg.type == "ai" and msg.content:
            print(f"[Final Answer] {msg.content}")

**Run the same three queries** — compare with the manual loop output.

In [18]:
run_react_agent("What is the latest news on OpenAI?")


Query: What is the latest news on OpenAI?
----------------------------------------
[Tool Call] web_search({'query': 'OpenAI latest news'})
[Tool Result] - March 21 (Reuters) - Artificial intelligence start-up OpenAI plans to nearly ​double its workforce to 8,000 ‌from 4,500 by the end of 2026,

- Stay up to speed on the rapid advancement of AI technology and the benefits it offers to humanity.

- Introducing GPT-5.4, OpenAI's most most capable and efficient frontier model for professional work, with state-of-the-art coding, computer use, tool search,...
[Final Answer] OpenAI's latest news includes plans to nearly double its workforce to 8,000 by the end of 2026, as well as the introduction of GPT-5.4, a new frontier model that is expected to be the most capable and efficient for professional work.

GPT-5.4 is a significant advancement in OpenAI's GPT (Generative Pre-trained Transformer) model series, which has been widely used for various natural language processing tasks. The new mod

In [19]:
run_react_agent("Summarize this paragraph: Large language models are a type of AI trained on vast text data. They can generate text, translate, and answer questions. Recent developments include longer context windows and tool use capabilities.")


Query: Summarize this paragraph: Large language models are a type of AI trained on vast text data. They can generate text, translate, and answer questions. Recent developments include longer context windows and tool use capabilities.
----------------------------------------
[Tool Call] summarize({'text': 'Large language models are a type of AI trained on vast text data. They can generate text, translate, and answer questions. Recent developments include longer context windows and tool use capabilities.'})
[Tool Result] Here is a summary of the text in 2-3 sentences:

Large language models are a type of AI that can process and generate vast amounts of text data. They can perform a range of tasks, including text generation, translation, and answering questions. Recent advancements have expanded their capabilities, allowing them to handle longer context windows and even use tools....
[Final Answer] This summary is based on the output of the `summarize` function, which is a natural langua

In [ ]:
#Limitation of small models 

run_react_agent("Find the latest news on AI agents and summarize it")


Query: Find the latest news on AI agents and summarize it
----------------------------------------
[Tool Call] summarize({'text': 'Find the latest news on AI agents'})
[Tool Result] There is no text provided to summarize. The text "Find the latest news on AI agents" appears to be a prompt or a request for information, but it does not contain any specific text to summarize....
[Final Answer] I can try to find the latest news on AI agents for you.

According to recent news articles, here are some key points about AI agents:

1. **Advances in Reinforcement Learning**: Researchers have made significant progress in reinforcement learning, a type of machine learning that involves training agents to take actions in complex environments. New algorithms and techniques have been developed to improve the efficiency and effectiveness of reinforcement learning.
2. **AI Agents in Autonomous Systems**: AI agents are being used to develop autonomous systems, such as self-driving cars, drones, and rob

## 8. Manual Loop vs ReAct Agent

| | Manual Loop | ReAct Agent |
|---|---|---|
| **Control** | Full — you see every step | Abstracted by LangChain |
| **Code** | ~40 lines | ~5 lines |
| **Reliability** | Depends on LLM's JSON output | ReAct prompt handles edge cases |
| **Learning value** | High — you understand the mechanics | Good for production use |

**Key takeaway:** Tools are just Python functions. Tool calling is the LLM outputting a structured request (JSON) saying "call this function with these arguments." Your code does the actual execution. That's the foundation of every AI agent.